In [3]:
import pandas as pd
import numpy as np

grid = pd.read_parquet('../data/processed/daily_sales.parquet')
print(f'Grid shape: {grid.shape}')

Grid shape: (1754, 15972)


In [4]:
VAL_DATES  = pd.date_range('2025-08-09', '2025-09-05', freq='D')
train_grid = grid[grid.index < '2025-08-09']
actual     = grid.loc[VAL_DATES]
weights    = train_grid.iloc[-28:].sum()

# Only SKUs that actually affect WRMSSE score
active_skus = weights[weights > 0].index.tolist()

print(f'Train period   : {train_grid.index[0].date()} → {train_grid.index[-1].date()}')
print(f'Val period     : {VAL_DATES[0].date()} → {VAL_DATES[-1].date()}')
print(f'Val days       : {len(VAL_DATES)}')
print(f'Total SKUs     : {len(grid.columns)}')
print(f'Active SKUs    : {len(active_skus)}  ← only these affect WRMSSE')

Train period   : 2020-11-17 → 2025-08-08
Val period     : 2025-08-09 → 2025-09-05
Val days       : 28
Total SKUs     : 15972
Active SKUs    : 2823  ← only these affect WRMSSE


In [5]:
def wrmsse(pred, actual, train, weights):
    # pred, actual : DataFrame (28 days × 15972 SKUs)
    mse        = ((pred.values - actual.values) ** 2).mean(axis=0)
    train_mean = train.iloc[-28:].mean(axis=0).values
    scale      = train_mean ** 2 + 1e-8   # avoid division by zero
    rmsse      = np.sqrt(mse / scale)
    w          = weights.values / weights.sum()
    return float((w * rmsse).sum())

In [6]:
pred_last = pd.DataFrame(
    np.tile(train_grid.iloc[-1].values, (len(VAL_DATES), 1)),
    index=VAL_DATES,
    columns=grid.columns
)

score_last = wrmsse(pred_last, actual, train_grid, weights)
print(f'WRMSSE — Last value : {round(score_last, 4)}')

WRMSSE — Last value : 2.5277


In [7]:
pred_mean = pd.DataFrame(
    np.tile(train_grid.iloc[-28:].mean().values, (len(VAL_DATES), 1)),
    index=VAL_DATES,
    columns=grid.columns
)

score_mean = wrmsse(pred_mean, actual, train_grid, weights)
print(f'WRMSSE — 28-day mean : {round(score_mean, 4)}')

WRMSSE — 28-day mean : 2.2522


In [8]:
def dow_forecast(series, dates, n_weeks=12):
    s     = series.copy()
    preds = []
    for d in dates:
        past = s[s.index.dayofweek == d.dayofweek].iloc[-n_weeks:]
        if len(past) > 0 and past.sum() > 0:
            w    = np.exp(np.linspace(-2, 0, len(past)))
            pred = float(np.average(past.values, weights=w))
        else:
            pred = float(s.iloc[-28:].mean())
        pred = max(0.0, pred)
        preds.append(pred)
        s[d] = pred   # recursive update
    return np.array(preds)

# ── Only loop over active SKUs (2,823 instead of 15,972) ──────
print(f'Running DOW forecast on {len(active_skus)} active SKUs...')

dow_preds = {}
for i, sku in enumerate(active_skus):
    dow_preds[sku] = dow_forecast(train_grid[sku], VAL_DATES)
    if (i + 1) % 500 == 0:
        print(f'  {i+1}/{len(active_skus)} done...')

# Fill inactive SKUs with zero
pred_dow = pd.DataFrame(0.0, index=VAL_DATES, columns=grid.columns)
for sku in active_skus:
    pred_dow[sku] = dow_preds[sku]

score_dow = wrmsse(pred_dow, actual, train_grid, weights)
print(f'WRMSSE — DOW avg : {round(score_dow, 4)}')

Running DOW forecast on 2823 active SKUs...
  500/2823 done...
  1000/2823 done...
  1500/2823 done...
  2000/2823 done...
  2500/2823 done...
WRMSSE — DOW avg : 2.4513


In [9]:
print('=' * 40)
print('BASELINE SUMMARY')
print('=' * 40)
print(f'Last value    : {round(score_last, 4)}')
print(f'28-day mean   : {round(score_mean, 4)}')
print(f'DOW avg       : {round(score_dow,  4)}  ← best baseline')
print('-' * 40)
print(f'Target for ML : beat {round(score_dow, 4)}')

BASELINE SUMMARY
Last value    : 2.5277
28-day mean   : 2.2522
DOW avg       : 2.4513  ← best baseline
----------------------------------------
Target for ML : beat 2.4513
